## Task 4

<div dir="rtl">


## 1 فهم البيانات

**المشكلة**

نبي نتنبأ اذا كان طلب القرض يوافق عليه او لا بناء على بيانات العميل والقرض

**حجم البيانات**

تحتوي البيانات على **614 صف** و **13 عمود** قبل التنظيف وانشاء ميزات جديدة

**الهدف (Target)**

الهدف هو `Loan_Status` في الملف الاصلي كانت القيمة **Y** تعني الموافقة على القرض بينما **N** تعني الرفض وبعد التنظيف حولناها الى **1** للموافقة و **0** للرفض عشان نسهل التحليل وبناء النموذج وبلغت نسبة الموافقة **68.73%** في البيانات

**الميزات (Features)**

تشمل الميزات الاصلية بيانات العميل والقرض مثل الدخل والحالة الاجتماعية والتعليم والسجل الائتماني ومبلغ القرض ومدة السداد اما `Loan_ID` فهو معرف للطلب ومو ميزة للتنبؤ وراح ننشئ ميزات جديدة بعدين مثل `Total_Income` و `Loan_to_Income` و `Payment_to_Income`

في هذا القسم بستعرض اول صفوف البيانات والاعمدة الموجودة عشان افهمها بشكل افضل

</div>

In [1]:
import pandas as pd
df = pd.read_csv("loan.csv")
print(df.shape)
print(df["Loan_Status"].unique())
print(df["Loan_Status"].value_counts(normalize=True) * 100)
df.head()

(614, 13)
['Y' 'N']
Loan_Status
Y    68.729642
N    31.270358
Name: proportion, dtype: float64


,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


## 2. EDA بالأرقام

<div dir="rtl">

#### توقعاتي قبل EDA

- اتوقع وجود بعض القيم المفقودة في `LoanAmount` و `Credit_History`
- اتوقع ما فيه صفوف مكررة لان لكل طلب `Loan_ID` مختلف
- اتوقع ان يكون `Credit_History` من اكثر الاعمدة ارتباطا بالموافقة على القرض

</div>

In [2]:
print(df.shape)
print(df.dtypes)
print(df.isnull().sum())
print(df.duplicated().sum())
df.describe()

(614, 13)
Loan_ID               object
Gender                object
Married               object
Dependents            object
Education             object
Self_Employed         object
ApplicantIncome        int64
CoapplicantIncome    float64
LoanAmount           float64
Loan_Amount_Term     float64
Credit_History       float64
Property_Area         object
Loan_Status           object
dtype: object
Loan_ID               0
Gender               13
Married               3
Dependents           15
Education             0
Self_Employed        32
ApplicantIncome       0
CoapplicantIncome     0
LoanAmount           22
Loan_Amount_Term     14
Credit_History       50
Property_Area         0
Loan_Status           0
dtype: int64
0


,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History
count,614.000000,614.000000,592.000000,600.00000,564.000000
mean,5403.459283,1621.245798,146.412162,342.00000,0.842199
std,6109.041673,2926.248369,85.587325,65.12041,0.364878
min,150.000000,0.000000,9.000000,12.00000,0.000000
25%,2877.500000,0.000000,100.000000,360.00000,1.000000
50%,3812.500000,1188.500000,128.000000,360.00000,1.000000
75%,5795.000000,2297.250000,168.000000,360.00000,1.000000
max,81000.000000,41667.000000,700.000000,480.00000,1.000000


<div dir="rtl">

#### ملاحظتي بعد التشغيل

تحتوي البيانات على **614 صف** و **13 عمود** حيث ان **8 اعمدة** من اصل 13 نصية (`object`) و **5 اعمدة** رقمية مثل `ApplicantIncome` و `LoanAmount` و `Credit_History`

كان توقعي ان فيه قيم مفقودة في `LoanAmount` و `Credit_History` وطلع توقعي صح جزئيا لان المجموع الكلي للقيم المفقودة كان **149 قيمة** وكان اكثر عمود فيه نقص هو `Credit_History` بـ **50 قيمة مفقودة** ثم `Self_Employed` بـ **32 قيمة** ثم `LoanAmount` بـ **22 قيمة**

وبعد توقعت ما يكون فيه صفوف مكررة وطلعت النتيجة **0 صفوف مكررة** لذلك كان هذا التوقع صح

من نتائج `describe` لاحظت ان متوسط `ApplicantIncome` يساوي **5403** بينما الوسيط **3812** بس واكبر قيمة وصلت الى **81000** وهذا يدل على وجود قيم مرتفعة رفعت المتوسط لفوق

ولاحظت بعد ان `Loan_Amount_Term` يتركز عند **360 شهر** لان الربع الاول والوسيط والربع الثالث كلها تساوي **360** وهذا يعني ان اغلب القروض في البيانات مدة سدادها **360 شهر**

</div>

<div dir="rtl">

### 3 التنظيف وسجل القرارات (Cleaning + Decision Log)
في هذا القسم نظفت البيانات وتعاملت مع القيم المفقودة وحولت بعض الاعمدة الى صيغ مناسبة للتحليل والنمذجة

وثقت كل قرار تنظيف مع السبب والاثر المتوقع في جدول Decision Log التالي

</div>

In [3]:
print("النقص قبل:", df.isna().sum().sum())
df_clean = df.copy()

for c in ["Gender", "Married", "Self_Employed"]:
    df_clean[c] = df_clean[c].fillna(df_clean[c].mode()[0])

df_clean["Dependents"] = (df_clean["Dependents"]
                          .fillna(df_clean["Dependents"].mode()[0])
                          .replace("3+", "3").astype(int))

df_clean["LoanAmount"] = df_clean["LoanAmount"].fillna(df_clean["LoanAmount"].median())
df_clean["Loan_Amount_Term"] = df_clean["Loan_Amount_Term"].fillna(df_clean["Loan_Amount_Term"].mode()[0])
df_clean["Credit_History"] = df_clean["Credit_History"].fillna(1)
df_clean["Loan_Status"] = df_clean["Loan_Status"].map({"Y": 1, "N": 0})

print("النقص بعد:", df_clean.isna().sum().sum())
print("الشكل:", df_clean.shape)
print(df_clean["Loan_Status"].value_counts())

النقص قبل: 149
النقص بعد: 0
الشكل: (614, 13)
Loan_Status
1    422
0    192
Name: count, dtype: int64


<div dir="rtl">

#### Decision Log

| القرار | السبب | الارقام |
|---|---|---|
| تعويض القيم المفقودة في `Gender` و `Married` و `Self_Employed` بالقيمة الاكثر تكرار | لانها اعمدة تصنيفية والقيم المفقودة فيها قليلة | Gender = 13 و Married = 3 و Self_Employed = 32 |
| تحويل القيمة `3+` في `Dependents` الى 3 وتعويض النقص بالـ Mode | عشان يصير العمود رقمي ويسهل تحليله | 51 قيمة `3+` و 15 قيمة مفقودة عوضت بـ 0 |
| تعويض القيم المفقودة في `LoanAmount` بالوسيط | لان الوسيط اقل تاثرا بالقيم المتطرفة من المتوسط | 22 قيمة مفقودة والوسيط = 128 والمتوسط = 146.4 |
| تعويض القيم المفقودة في `Loan_Amount_Term` بـ 360 | لانها القيمة الاكثر تكرار في العمود | 14 قيمة مفقودة و 360 تكررت 512 مرة |
| تعويض القيم المفقودة في `Credit_History` بـ 1 | لان نسبة الموافقة للمفقودين كانت اقرب لمجموعة 1 | 50 صف مفقود و 74% للمفقود و 79.6% لمجموعة 1 و 7.9% لمجموعة 0 |
| الابقاء على القيم المتطرفة | لانها قيم معقولة ومو اخطاء في البيانات | ApplicantIncome = 50 و LoanAmount = 41 و CoapplicantIncome = 18 |
| تحويل `Loan_Status` من Y/N الى 1/0 | لان النماذج تتعامل مع الارقام بشكل افضل | 422 قيمة = 1 و 192 قيمة = 0 |
| الابقاء على `Loan_ID` | لان المطلوب في المهمة يمنع حذف الاعمدة بالوقت الحالي | 614 قيمة فريدة و 0 قيم مفقودة |

</div>

<div dir="rtl">

#### ملاحظتي بعد التشغيل

نتائج التنظيف طلعت مطابقة للي وصلت له في `day11` حيث انخفض عدد القيم المفقودة من **149 قيمة** الى **0** بعد ما طبقت خطوات التنظيف وهذا يعني ان البيانات صارت خالية من القيم المفقودة وجاهزة للتحليل

وبقي حجم البيانات **614 صف** و **13 عمود** لاننا عالجنا القيم المفقودة بدون ما نحذف اي سجلات من البيانات

وبعد ما حولت `Loan_Status` من **Y/N** الى **1/0** صار توزيع الهدف **422 حالة موافقة** و **192 حالة رفض** وهو نفس التوزيع اللي سجلته قبل

</div>

<div dir="rtl">

#### 4 هندسة الميزات (Feature Engineering)

في هذا القسم انشات ميزات جديدة عشان امثل القدرة المالية للعميل بشكل افضل ونختبر اذا تحسن جودة التنبؤ

- `Total_Income`: مجموع دخل المتقدم ودخل الشريك عشان يمثل الدخل الكلي للاسرة
- `Has_Coapplicant`: يوضح اذا كان فيه شريك في الطلب (**1**) او ما فيه (**0**)
- `Loan_to_Income`: يقارن مبلغ القرض (بعد ضرب `LoanAmount` في **1000** لانه بالالاف) بالدخل الكلي عشان نعرف حجم القرض بالنسبة لدخل العميل
- `Monthly_Payment`: يمثل القسط الشهري التقريبي للقرض (بعد ضرب `LoanAmount` في **1000** وقسمته على مدة السداد)
- `Payment_to_Income`: يقارن القسط الشهري بالدخل الكلي ويعكس عبء السداد على العميل
- `Payment_bin`: تقسيم `Payment_to_Income` الى اربع فئات متساوية العدد عشان يسهل علينا مقارنة نسب الموافقة
- `Income_bin`: تقسيم `ApplicantIncome` الى اربع فئات متساوية العدد عشان ندرس اثر الدخل على الموافقة

</div>

In [4]:
df_feat = df_clean.copy()

df_feat["Total_Income"] = df_feat["ApplicantIncome"] + df_feat["CoapplicantIncome"]
df_feat["Has_Coapplicant"] = (df_feat["CoapplicantIncome"] > 0).astype(int)
df_feat["Loan_to_Income"] = (df_feat["LoanAmount"] * 1000) / df_feat["Total_Income"]
df_feat["Monthly_Payment"] = (df_feat["LoanAmount"] * 1000) / df_feat["Loan_Amount_Term"]
df_feat["Payment_to_Income"] = df_feat["Monthly_Payment"] / df_feat["Total_Income"]

labels = ["Q1 (الأقل)", "Q2", "Q3", "Q4 (الأعلى)"]
df_feat["Payment_bin"] = pd.qcut(df_feat["Payment_to_Income"], 4, labels=labels)
df_feat["Income_bin"] = pd.qcut(df_feat["ApplicantIncome"], 4, labels=labels)

print("الشكل:", df_feat.shape)
new_cols = ["Total_Income", "Has_Coapplicant", "Loan_to_Income", "Monthly_Payment",
            "Payment_to_Income", "Payment_bin", "Income_bin"]
df_feat[new_cols].head()

الشكل: (614, 20)


,Total_Income,Has_Coapplicant,Loan_to_Income,Monthly_Payment,Payment_to_Income,Payment_bin,Income_bin
0,5849.0,0,21.884083,355.555556,0.060789,Q2,Q4 (الأعلى)
1,6091.0,1,21.014612,355.555556,0.058374,Q2,Q3
2,3000.0,0,22.000000,183.333333,0.061111,Q2,Q2
3,4941.0,1,24.286582,333.333333,0.067463,Q2,Q1 (الأقل)
4,6000.0,0,23.500000,391.666667,0.065278,Q2,Q4 (الأعلى)


<div dir="rtl">

#### ملاحظتي بعد التشغيل

صار شكل البيانات **(614, 20)** يعني **614 صف** و **20 عمود** بعد ما بدانا بـ **13 عمود** اصلي واضفنا **7 ميزات جديدة** خلال مرحلة هندسة الميزات

ومن مراجعة الصف الاول كان القسط الشهري (`Monthly_Payment`) يساوي **355.56** بينما كانت قيمة `Payment_to_Income` تساوي **0.0608** وهذا يعني ان القسط يمثل تقريبا **6.1%** من الدخل الكلي وهي نسبة منخفضة نسبيا

ولاحظت بعد ان القيم الجديدة تبدو منطقية ومتوافقة مع المعادلات المستخدمة وهذا يعطيني ثقة بان الميزات المولدة تم حسابها بشكل صحيح

</div>

### 5 اختيار الميزات (Feature Selection)

<div dir="rtl">

في هذا القسم بستخدم نتائج التحليل لاختيار افضل الميزات للتنبؤ بالموافقة على القرض بقارن بين الاعمدة التصنيفية من خلال حساب الفجوة بين اعلى واقل نسبة موافقة وبقارن بين الاعمدة الرقمية باستخدام الارتباط مع `Loan_Status`

وبفحص بعد التكرار بين بعض الاعمدة مثل `ApplicantIncome` و `Total_Income` عشان اعرف اذا كانت تعطي نفس المعلومة تقريبا وفي النهاية بعتمد على هذي النتائج لبناء جدول القرار النهائي (`Keep/Drop`) واختيار الميزات الاكثر فائدة للموديل

</div>

In [5]:
cat_cols = ["Gender", "Married", "Dependents", "Education", "Self_Employed",
            "Credit_History", "Property_Area", "Has_Coapplicant", "Payment_bin", "Income_bin"]

results = []
for col in cat_cols:
    rates = df_feat.groupby(col, observed=False)["Loan_Status"].mean() * 100
    results.append({"column": col, "max_rate": rates.max(), "min_rate": rates.min(),
                    "gap": rates.max() - rates.min()})
gap_table = pd.DataFrame(results).sort_values("gap", ascending=False)
print(gap_table)

num_cols = df_feat.select_dtypes(include="number").columns
corr_table = (df_feat[num_cols].corr()["Loan_Status"].drop("Loan_Status")
              .sort_values(key=abs, ascending=False))
print(corr_table)

print("ApplicantIncome vs Total_Income:", df_feat["ApplicantIncome"].corr(df_feat["Total_Income"]))

            column   max_rate   min_rate        gap
5   Credit_History  79.047619   7.865169  71.182451
6    Property_Area  76.824034  61.452514  15.371520
2       Dependents  75.247525  64.705882  10.541642
3        Education  70.833333  61.194030   9.639303
8      Payment_bin  73.856209  64.285714   9.570495
1          Married  71.820449  62.910798   8.909651
7  Has_Coapplicant  71.847507  64.835165   7.012342
0           Gender  69.123506  66.964286   2.159220
9       Income_bin  69.281046  68.181818   1.099228
4    Self_Employed  68.796992  68.292683   0.504310
Credit_History       0.540556
Loan_to_Income      -0.086141
Has_Coapplicant      0.075165
CoapplicantIncome   -0.059187
Payment_to_Income   -0.051361
LoanAmount          -0.033214
Total_Income        -0.031271
Loan_Amount_Term    -0.022549
Monthly_Payment     -0.011757
Dependents           0.010118
ApplicantIncome     -0.004710
Name: Loan_Status, dtype: float64
ApplicantIncome vs Total_Income: 0.8930372940684422


<div dir="rtl">

## جدول القرار النهائي

| العمود | القرار | السبب | الرقم |
|---|---|---|---|
| `Loan_ID` | Drop | معرف فريد لكل طلب وما يساعد في التنبؤ | 614 قيمة مختلفة |
| `Gender` | Drop | الفجوة ضعيفة وممكن يسبب مشاكل عدالة في قرارات القروض | Gap = 2.16 |
| `ApplicantIncome` | Drop | مكرر تقريبا مع Total_Income | Corr = 0.893 مع Total_Income |
| `Total_Income` | Keep | يمثل الدخل الكلي وارتباطه اقوى من ApplicantIncome | Corr = -0.03 |
| `Credit_History` | Keep | اقوى عمود في البيانات كلها | Gap = 71.18 و Corr = 0.54 |
| `Married` | Keep | له فجوة ملحوظة في نسبة الموافقة | Gap = 8.91 |
| `Dependents` | Keep | له فجوة جيدة وتاثر على القرار | Gap = 10.54 |
| `Education` | Keep | له فجوة جيدة في نسبة الموافقة | Gap = 9.64 |
| `Self_Employed` | Drop | تاثيره ضعيف جدا | Gap = 0.50 |
| `Property_Area` | Keep | من اقوى الاعمدة التصنيفية | Gap = 15.37 |
| `Has_Coapplicant` | Keep | له تاثير واضح على الموافقة | Gap = 7.01 |
| `CoapplicantIncome` | Drop | معلوماته ممثلة اصلا ضمن Total_Income | Corr = -0.06 |
| `LoanAmount` | Drop | ارتباط ضعيف لحاله | Corr = -0.03 |
| `Loan_Amount_Term` | Drop | ارتباط ضعيف جدا | Corr = -0.02 |
| `Loan_to_Income` | Keep | ثاني اقوى عمود رقمي بعد Credit_History وارتباطه مع Payment_to_Income هو 0.31 لذلك مو مكررين | Corr = -0.09 |
| `Monthly_Payment` | Drop | ارتباط ضعيف جدا | Corr = -0.01 |
| `Payment_to_Income` | Keep | اظهر فرق واضح بين المجموعات ويقيس العبء الشهري | Gap (Payment_bin) = 9.57 و Corr = -0.05 |
| `Payment_bin` | Drop | نسخة مصنفة ومكررة من Payment_to_Income | Gap = 9.57 |
| `Income_bin` | Drop | نسخة مصنفة من الدخل وتاثيره ضعيف | Gap = 1.10 |

### القاعدة

احذف الاعمدة المعرفة او المكررة او اللي تاثيرها ضعيف جدا وابقي الاعمدة اللي اظهرت ارتباط او فجوة واضحة وممكن تضيف معلومة جديدة للموديل

### الخلاصة

بعد ما راجعت الارتباطات والفجوات والتكرار بين الميزات ابقيت **9 اعمدة** وحذفت **10 اعمدة**

</div>

## 6 حفظ الداتا

In [6]:
selected = ["Total_Income", "Credit_History", "Married", "Dependents", "Education",
            "Property_Area", "Has_Coapplicant", "Loan_to_Income", "Payment_to_Income"]
print("عدد الميزات النهائية:", len(selected))
print(selected)

عدد الميزات النهائية: 9
['Total_Income', 'Credit_History', 'Married', 'Dependents', 'Education', 'Property_Area', 'Has_Coapplicant', 'Loan_to_Income', 'Payment_to_Income']


<div dir="rtl">

#### توقعاتي للملف النهائي

اتوقع ان الملف النهائي بيحتوي على **614 صف** و **9 او 10 اعمدة** والاقرب **10 اعمدة**

</div>

In [7]:
model_ready = df_feat[selected + ["Loan_Status"]]
model_ready.to_csv("model_ready.csv", index=False)
print(model_ready.shape)

(614, 10)


<div dir="rtl">

#### ملاحظتي بعد التشغيل

كان توقعي ان الملف النهائي بيحتوي على **614 صف** و **9 او 10 اعمدة** والاقرب **10 اعمدة** والنتيجة الفعلية طلعت **(614, 10)** لذلك كان توقعي صح

بقي عدد الصفوف **614 صف** بدون تغيير لاننا ما حذفنا اي سجل من البيانات بس ركزنا على اختيار افضل الميزات وعدد الاعمدة صار **10 اعمدة** مكونة من **9 ميزات** مختارة بعناية بالاضافة الى `Loan_Status` كمتغير الهدف

اشوف ان هذي النتيجة منطقية لان هدف `Feature Selection` كان تقليل عدد الاعمدة مع الحفاظ على المعلومات المهمة لذلك انتهينا بملف اصغر واكثر تركيز بدون ما نفقد اي سجلات

وتم حفظ الملف بنجاح باسم `model_ready.csv` وصار جاهز للاستخدام في بناء النموذج وتقييمه

</div>

### 7 التقرير النهائي

<div dir="rtl">

#### 7 التقرير النهائي

#### الاستنتاجات

1. تحتوي البيانات على **614 صف** و **13 عمود** وكان فيها **149 قيمة مفقودة** قبل التنظيف منها **50 قيمة مفقودة** في `Credit_History` وعالجنا كل القيم المفقودة بدون ما نحذف اي سجل

2. بلغت نسبة الموافقة على القروض **68.73%** حيث تمت الموافقة على **422 طلب** مقابل **192 طلب** مرفوض

3. كان `Credit_History` اقوى عامل مرتبط بالموافقة على القرض حيث بلغت نسبة الموافقة **79.05%** لاصحاب السجل الائتماني الجيد مقابل **7.87%** للفئة الثانية بفجوة **71.18 نقطة مئوية** وارتباط بلغ **0.54** مع `Loan_Status`

4. اظهر `Property_Area` تاثير ملحوظ على الموافقة بفجوة **15.37 نقطة مئوية** بينما كان تاثير `Self_Employed` ضعيف جدا بفجوة **0.50 نقطة مئوية** بس

5. بلغ الارتباط بين `ApplicantIncome` و `Total_Income` قيمة **0.893** لذلك احتفظت بـ `Total_Income` فقط اما `Loan_to_Income` و `Payment_to_Income` فكان الارتباط بينهم **0.31** بس وهذا يدل على انهم يقدمون معلومات مختلفة ومو مكررين

#### مقارنة الفرضيات بالنتائج

لما قارنت فرضياتي بالنتائج الفعلية لقيت ان بعض التوقعات كانت صح وبعضها ما كان دقيق توقعت ان يكون `Credit_History` من اهم العوامل المؤثرة وطلع توقعي صح حيث حقق اعلى فجوة (**71.18 نقطة مئوية**) واعلى ارتباط (**0.54**) وتوقعت بعد ان تاثير `Self_Employed` يكون ضعيف واكدت النتائج هذا الشي بفجوة بلغت **0.50** بس

في المقابل توقعت ان `ApplicantIncome` يكون عامل مهم بس ارتباطه مع `Loan_Status` كان ضعيف جدا (**-0.005**) وتوقعت بعد ان `LoanAmount` يكون اكثر تاثير مما ظهر فعليا بس ارتباطه طلع **-0.03** بس واكثر نتيجة فاجاتني كانت `Property_Area` حيث ظهر تاثيره اوضح من المتوقع بفجوة بلغت **15.37 نقطة مئوية**

#### توصية للبنك

اوصي البنك بالتركيز بشكل اكبر على `Credit_History` عند تقييم طلبات القروض لانه كان المتغير الاكثر تاثير في البيانات حيث بلغت نسبة الموافقة **79.05%** لاصحاب السجل الائتماني الجيد مقابل **7.87%** بس للفئة الثانية بفجوة بلغت **71.18 نقطة مئوية** واوصي بعد بمراجعة الطلبات اللي كانت قيمة `Credit_History` فيها مفقودة قبل التنظيف (**50 حالة**) لاننا عوضناها بالقيمة **1** ومن الافضل التاكد من سجلها الائتماني الفعلي قبل الاعتماد عليها في اتخاذ القرار

#### رايي النهائي في Gender

اشوف ان من الافضل ما نستخدم `Gender` في النموذج النهائي لان تاثيره كان محدود جدا بفجوة بلغت **2.16 نقطة مئوية** بس وبالاضافة الى هذا ممكن يؤدي استخدامه الى قرارات غير عادلة او متحيزة بدون ما يقدم فائدة تنبؤية كبيرة وكمان عدد النساء في البيانات (**112**) اقل بكثير من عدد الرجال (**489**) وهذا ممكن يزيد احتمالية التحيز لذلك افضل استبعاده من النموذج والاعتماد على متغيرات اكثر ارتباط بالقدرة الفعلية على السداد

</div>